# Task 3: Qwen2.5-Coder-7B QLoRA SFT & Debugging Pipeline

Pipeline for Vietnamese Pedagogical Feedback Generation (Task 3).
- **Architecture**: Qwen2.5-Coder-7B-Instruct with 4-bit NormalFloat (NF4) double quantization.
- **Data Strategy**: Mixed balanced fine-tuning (real student submissions + verified synthetic pairs).
- **Execution Mode**: `DEBUG_MODE=True` for rapid 15-step smoke test and real-val slice validation; `DEBUG_MODE=False` for production training and 328-sample public inference.


In [ ]:
# ==============================================================================
# Cell 1: Pipeline Configuration & Execution Flags
# ==============================================================================
import os

# Repository & VCS configuration
REPO_URL = "github.com/tiendinhquang2104/llm-final-project.git"
BRANCH = "fix/task3-strict-feedback"
WORK_DIR = "/kaggle/tmp"

# Execution mode: True = Smoke test 15 steps + real-val eval; False = Full training + public eval
DEBUG_MODE: bool = False
FORCE_RETRAIN: bool = False
SMOKE_STEPS: int = 15

# Experiment settings
EXPERIMENT: str = "mixed_balanced"
EPOCHS: int = 2
MAX_STEPS: int = SMOKE_STEPS if DEBUG_MODE else None
EVAL_SPLIT: str = "real-val" if DEBUG_MODE else "public"

# Dataset sources (fallback Google Drive IDs)
DRIVE_ID_DATA = "1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj"
DRIVE_ID_SYNTH = "1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl"

print(f"[CONFIG] Mode: {'DEBUG' if DEBUG_MODE else 'PRODUCTION'} | Split: {EVAL_SPLIT} | Epochs: {EPOCHS} | Steps: {MAX_STEPS}")

In [ ]:
# ==============================================================================
# Cell 2: Workspace Initialization & Environment Setup
# ==============================================================================
import subprocess, sys, os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

def resolve_secret(key: str) -> str:
    try:
        return UserSecretsClient().get_secret(key) or ""
    except Exception:
        return ""

token = resolve_secret("GITHUB_TOKEN")
clone_url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"

repo_path = Path(WORK_DIR) / "repo"
if not repo_path.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url, str(repo_path)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Propagate credentials
env_vars = {}
for secret_key in ["GEMINI_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]:
    val = resolve_secret(secret_key)
    if val:
        os.environ[secret_key] = val
        env_vars[secret_key] = val

if env_vars:
    env_content = "".join(f"{k}={v}\n" for k, v in env_vars.items())
    Path(".env").write_text(env_content, encoding="utf-8")

head_commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print(f"[ENV] Workspace: {os.getcwd()} | Current commit: {head_commit}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies Installation & CUDA Hardware Sanity
# ==============================================================================
!pip install -q gdown pyyaml pytest
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes peft trl datasets

import torch
assert torch.cuda.is_available(), "CUDA runtime is required."
device_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[HARDWARE] GPU 0: {device_name} | VRAM: {total_vram:.2f} GiB | Visible GPUs: {torch.cuda.device_count()}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader


In [ ]:
# ==============================================================================
# Cell 4: Data Ingestion & Preprocessing
# ==============================================================================
import os, shutil, glob, gdown, subprocess, zipfile, json
from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/synth").mkdir(parents=True, exist_ok=True)
download_dir = Path(WORK_DIR) / "download"
download_dir.mkdir(parents=True, exist_ok=True)

def safe_extract_zip(zip_path: Path, target_dir: Path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)

# 1. Resolve raw datasets (sample_dataset and public_test_release)
sample_found = glob.glob("/kaggle/input/**/sample_dataset", recursive=True)
public_found = glob.glob("/kaggle/input/**/public_test_release", recursive=True)

if sample_found and not Path("data/raw/sample_dataset").exists():
    shutil.copytree(sample_found[0], "data/raw/sample_dataset")
if public_found and not Path("data/raw/public_test_release").exists():
    shutil.copytree(public_found[0], "data/raw/public_test_release")

if not (Path("data/raw/sample_dataset/exams.json").exists() and Path("data/raw/public_test_release/exams.json").exists()):
    zip_dest = download_dir / "data.zip"
    if not zip_dest.exists() and DRIVE_ID_DATA:
        gdown.download(id=DRIVE_ID_DATA, output=str(zip_dest), quiet=False)
    
    if zip_dest.exists():
        safe_extract_zip(zip_dest, Path("data/raw"))
    # Unpack any bundle or nested zip archives
    for z in list(Path("data/raw").glob("*.zip")) + list(download_dir.glob("*.zip")):
        try:
            safe_extract_zip(z, Path("data/raw"))
        except Exception:
            pass

# Flatten any nested split directories
for split_name in ["sample_dataset", "public_test_release"]:
    nested = Path(f"data/raw/{split_name}/{split_name}")
    if nested.exists() and nested.is_dir():
        for item in nested.iterdir():
            shutil.move(str(item), str(nested.parent))
        nested.rmdir()

shutil.rmtree("data/raw/__MACOSX", ignore_errors=True)

# 2. Resolve synthetic dataset
synth_target = Path("data/synth/generate-dataset")
synth_found = [d for d in glob.glob("/kaggle/input/**/generate-dataset*", recursive=True) if os.path.isdir(d) and (Path(d) / "exams.json").exists()]

if synth_found and not synth_target.exists():
    shutil.copytree(synth_found[0], synth_target)
elif not synth_target.exists() and DRIVE_ID_SYNTH:
    synth_zip = download_dir / "synth.zip"
    if not synth_zip.exists():
        gdown.download(id=DRIVE_ID_SYNTH, output=str(synth_zip), quiet=False)
    if synth_zip.exists():
        safe_extract_zip(synth_zip, Path("data/synth"))
        nested_synth = Path("data/synth/generate-dataset/generate-dataset")
        if nested_synth.exists() and nested_synth.is_dir():
            for item in nested_synth.iterdir():
                shutil.move(str(item), str(nested_synth.parent))
            nested_synth.rmdir()

shutil.rmtree("data/synth/__MACOSX", ignore_errors=True)

# 3. Ensure public requested levels mapping exists
req_levels_path = Path("data/raw/public_test_release/task3_requested_levels.json")
if not req_levels_path.exists() and Path("data/raw/public_test_release/task3_feedback.json").exists():
    with open("data/raw/public_test_release/task3_feedback.json", "r", encoding="utf-8-sig") as f:
        public_fb = json.load(f)
    req_map = {
        s["sample_id"]: s["input"]["target_feedback_level"]
        for s in public_fb.get("samples", [])
        if "sample_id" in s and "input" in s and "target_feedback_level" in s["input"]
    }
    with open(req_levels_path, "w", encoding="utf-8") as f:
        json.dump(req_map, f, indent=2, ensure_ascii=False)
    print(f"[DATA] Created public requested levels: {len(req_map)} entries.")

# 4. Verify and execute Task 3 dataset preparation
assert Path("data/raw/sample_dataset/exams.json").exists(), "Missing data/raw/sample_dataset/exams.json"
assert Path("data/raw/public_test_release/exams.json").exists(), "Missing data/raw/public_test_release/exams.json"
assert req_levels_path.exists(), "Missing task3_requested_levels.json"

subprocess.run(["python", "scripts/task3_prepare.py", "--filter-synthetic", "strict"], check=True)
print("[DATA] Partitions prepared successfully.")


In [ ]:
# ==============================================================================
# Cell 5: QLoRA Training + Stage-2 Real-Train Curriculum Calibration
# ==============================================================================
import yaml, shutil, subprocess, os, glob, json
from pathlib import Path

adapter_dir = f"checkpoints/task3/{EXPERIMENT}"
stage2_dir = "checkpoints/task3/stage2_real_calibrated"
working_adapter = "/kaggle/working/task3_lora_adapter"
STAGE2_CALIBRATE_REAL = True

if FORCE_RETRAIN:
    shutil.rmtree(working_adapter, ignore_errors=True)
    shutil.rmtree(adapter_dir, ignore_errors=True)
    shutil.rmtree(stage2_dir, ignore_errors=True)

input_adapters = [
    p for p in glob.glob("/kaggle/input/**/task3_lora_adapter", recursive=True)
    if (Path(p) / "adapter_model.safetensors").exists() or (Path(p) / "adapter_model.bin").exists()
]

checkpoint_exists = any(
    Path(p).exists() for p in [
        f"{working_adapter}/adapter_model.safetensors",
        f"{working_adapter}/adapter_model.bin",
        f"{adapter_dir}/adapter_model.safetensors",
        f"{adapter_dir}/adapter/adapter_model.safetensors",
    ]
) or bool(input_adapters)

train_env = os.environ.copy()
train_env["CUDA_VISIBLE_DEVICES"] = "0"
train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if checkpoint_exists and not DEBUG_MODE and not FORCE_RETRAIN:
    if input_adapters and not Path(f"{working_adapter}/adapter_model.safetensors").exists():
        shutil.copytree(input_adapters[0], working_adapter, dirs_exist_ok=True)
        print(f"[TRAIN] Reusing mounted Stage-1 adapter from {input_adapters[0]} -> {working_adapter}.")
    else:
        print("[TRAIN] Existing Stage-1 adapter detected.")
else:
    cfg_path = Path("configs/task3/qlora.yaml")
    cfg_path.write_text(yaml.safe_dump({
        "extends": "baseline.yaml",
        "experiment": EXPERIMENT,
        "output_dir": adapter_dir,
        "min_free_vram_gib": 6,
        "attn_implementation": "sdpa",
        "max_seq_length": 4096,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "optim": "paged_adamw_8bit",
        "learning_rate": 0.00002,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "max_grad_norm": 1.0,
        "epochs": EPOCHS,
        "gradient_accumulation_steps": 4,
        "source_weights": {"real": 0.65, "synthetic": 0.35},
        "prompt_policy_version": 3,
        "attach_compiler_evidence": True,
        "cpp_standard": "c++11",
        "ignore_competing_gpu_workload": True,
        "device_map": "auto"
    }), encoding="utf-8")

    train_cmd = [
        "python", "scripts/train_task3.py",
        "--config", str(cfg_path),
        "--experiment", EXPERIMENT,
        "--output-dir", adapter_dir
    ]
    if MAX_STEPS:
        train_cmd.extend(["--max-steps", str(MAX_STEPS)])

    print("[TRAIN] Executing Stage-1:", " ".join(train_cmd))
    subprocess.run(train_cmd, env=train_env, check=True)

    source_dir = f"{adapter_dir}/adapter" if Path(f"{adapter_dir}/adapter").exists() else adapter_dir
    shutil.copytree(source_dir, working_adapter, dirs_exist_ok=True)
    print(f"[TRAIN] Stage-1 Adapter synchronized to {working_adapter}")

# Stage-2 Real-Train Curriculum Calibration (~2 mins on T4: aligns style to real instructor feedback)
stage2_marker = Path(working_adapter) / "stage2_real_calibrated.json"
if STAGE2_CALIBRATE_REAL and not DEBUG_MODE and not stage2_marker.exists():
    shutil.rmtree(stage2_dir, ignore_errors=True)
    stage2_cfg = Path("configs/task3/qlora_stage2_real.yaml")
    stage2_cfg.write_text(yaml.safe_dump({
        "extends": "baseline.yaml",
        "experiment": "real",
        "output_dir": stage2_dir,
        "staged_adapter": working_adapter,
        "stage_name": "stage2-real-instructor-calibration",
        "min_free_vram_gib": 6,
        "attn_implementation": "sdpa",
        "max_seq_length": 4096,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "optim": "paged_adamw_8bit",
        "learning_rate": 0.000008,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "max_grad_norm": 1.0,
        "epochs": 2,
        "gradient_accumulation_steps": 2,
        "prompt_policy_version": 3,
        "attach_compiler_evidence": True,
        "cpp_standard": "c++11",
        "ignore_competing_gpu_workload": True,
        "device_map": "auto"
    }), encoding="utf-8")
    stage2_cmd = [
        "python", "scripts/train_task3.py",
        "--config", str(stage2_cfg),
        "--experiment", "real",
        "--output-dir", stage2_dir,
        "--max-steps", "14"
    ]
    print("[TRAIN] Executing Stage-2 Real-Train Curriculum Calibration (14 steps on real instructor data):", " ".join(stage2_cmd))
    subprocess.run(stage2_cmd, env=train_env, check=True)
    s2_src = f"{stage2_dir}/adapter" if Path(f"{stage2_dir}/adapter").exists() else stage2_dir
    shutil.copytree(s2_src, working_adapter, dirs_exist_ok=True)
    stage2_marker.write_text(json.dumps({"stage2_calibrated": True, "steps": 14, "lr": 8e-6}), encoding="utf-8")
    print(f"[TRAIN] Stage-2 Real-Calibrated Adapter saved to {working_adapter}")

In [ ]:
# ==============================================================================
# Cell 6: Bounded Batch Inference (Counterexample Witness + AST P1-P4 + Hybrid RAG)
# ==============================================================================
import os, re, json, shutil, subprocess, tempfile, glob, yaml
from collections import Counter
from pathlib import Path
from src.synth.taxonomy import LABEL_SPACE

mode_str = "debug" if DEBUG_MODE else "prod"
output_file = f"outputs/task3/{mode_str}_{EVAL_SPLIT}.predictions.json"
Path("outputs/task3").mkdir(parents=True, exist_ok=True)

SINGLE_PROBLEM_SUITES = {
    "Class03-Midterm-1": {
        "normal": [("153\n", "True"), ("1634\n", "True"), ("123\n", "False"), ("370\n", "True"), ("9474\n", "True"), ("9475\n", "False")],
        "edge": [("1\n", "True"), ("9\n", "True"), ("10\n", "False"), ("100\n", "False"), ("1000000000\n", "False"), ("534494836\n", "True")],
    },
    "Class03-Midterm-2": {
        "normal": [
            ("16\n1 5 3 7 2 3 2 5 2 6 5 6 5 7 2 3\n", "6 1 5 3 7 2 6"),
            ("5\n5 5 5 5 5\n", "1 5"),
            ("6\n1 2 3 2 1 4\n", "4 1 2 3 4"),
        ],
        "edge": [
            ("1\n42\n", "1 42"),
            ("4\n1 2 2 2\n", "2 1 2"),
            ("5\n-1000000000 1000000000 -1000000000 0 0\n", "3 -1000000000 1000000000 0"),
        ],
    },
    "Class03-Final-1": {
        "normal": [
            ("harry potter\n", "hap"),
            ("tom riddle\n", "tomr"),
            ("abacaba abacaba\n", "aa"),
            ("cba d\n", "cbad"),
        ],
        "edge": [
            ("a a\n", "aa"),
            ("z a\n", "za"),
            ("bb b\n", "bb"),
            ("ba b\n", "bab"),
        ],
    },
}

def strip_cpp_comments(code: str) -> str:
    return re.sub(r"//[^\n]*|/\*[\s\S]*?\*/", "", code)

def evaluate_single_problem_runtime(code_path: Path, exam_id: str):
    suite = SINGLE_PROBLEM_SUITES.get(exam_id)
    if not suite or not shutil.which("g++"):
        return [], None
    tmp = Path(tempfile.mkdtemp())
    exe = tmp / "student_bin"
    try:
        c_res = subprocess.run(
            ["g++", "-std=c++11", str(code_path), "-o", str(exe)],
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=8
        )
        if c_res.returncode != 0:
            if re.search(r"^\S+:\d+:\d+: (?:fatal )?error:", c_res.stderr or "", re.M):
                return ["Lỗi biên dịch"], {"status": "compile_error"}
            return ["Lỗi logic"], {"status": "link_error", "detail": (c_res.stderr or "")[:200]}
        norm_pass = 0
        edge_pass = 0
        io_mismatch = False
        first_fail = None
        for kind, cases in suite.items():
            for inp, exp in cases:
                try:
                    r_res = subprocess.run(
                        [str(exe)], input=inp,
                        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=1.5
                    )
                    if r_res.returncode != 0:
                        return ["Lỗi logic"], {"status": "runtime_crash", "input": inp.strip(), "returncode": r_res.returncode}
                    raw_out = (r_res.stdout or "").strip()
                    cleaned = re.sub(r"^(?:nhap|input)[^:\n]*:\s*", "", raw_out, flags=re.I).strip()
                    toks = " ".join(cleaned.split())
                    exp_toks = " ".join(exp.split())
                    if toks == exp_toks:
                        if kind == "normal":
                            norm_pass += 1
                        else:
                            edge_pass += 1
                    else:
                        if first_fail is None:
                            first_fail = {"kind": kind, "input": inp.strip(), "expected": exp_toks, "actual": toks[:80]}
                        if toks.lower() == exp_toks.lower() or (toks.endswith(exp_toks) and len(toks) > len(exp_toks)):
                            io_mismatch = True
                except subprocess.TimeoutExpired:
                    return ["Lỗi vòng lặp", "Lỗi logic"], {"status": "timeout_infinite_loop", "input": inp.strip()}
                except Exception:
                    return ["Lỗi logic"], {"status": "runtime_error"}
        total_cases = len(suite["normal"]) + len(suite["edge"])
        witness = {
            "status": "all_passed" if (norm_pass + edge_pass == total_cases) else "failed_test_cases",
            "passed": norm_pass + edge_pass,
            "total": total_cases,
            "first_failed_case": first_fail,
        }
        if norm_pass == len(suite["normal"]) and edge_pass == len(suite["edge"]):
            return [], witness
        if io_mismatch and norm_pass + edge_pass >= len(suite["normal"]) - 1:
            return ["Lỗi nhập/xuất"], witness
        if norm_pass == len(suite["normal"]) and edge_pass < len(suite["edge"]):
            return ["Lỗi edge case"], witness
        return ["Lỗi logic"], witness
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def analyze_multi_problem_ast(code: str, is_compile_err: bool, compile_stderr: str) -> dict:
    stripped = strip_cpp_comments(code)
    status_by_prob = {}
    for idx, p_name in enumerate(("prob1", "prob2", "prob3", "prob4"), start=1):
        pid = f"P{idx}"
        m = re.search(rf"\b{p_name}\s*\([^)]*\)\s*\{{", stripped)
        if not m:
            status_by_prob[pid] = "missing_or_renamed"
            continue
        start = m.end()
        depth = 1
        pos = start
        while pos < len(stripped) and depth > 0:
            if stripped[pos] == "{":
                depth += 1
            elif stripped[pos] == "}":
                depth -= 1
            pos += 1
        body = stripped[start : max(start, pos - 1)].strip()
        if not body:
            status_by_prob[pid] = "not_implemented_empty_body"
        elif pid == "P1" and re.search(r'\b(?:(?:ifstream|fstream)\s+[A-Za-z_]\w*\s*\(\s*"[^"]+"|(?:open|ifstream|fstream|freopen)\s*\(\s*"[^"]+")', body):
            status_by_prob[pid] = "hardcoded_filename_instead_of_parameter"
        elif pid in ("P1", "P2", "P3") and "return" not in body:
            status_by_prob[pid] = "missing_return_statement"
        else:
            status_by_prob[pid] = "implemented"
    return {
        "compiler_status": "compile_error" if is_compile_err else "compiled",
        "problem_ast_status": status_by_prob,
        "prerequisite_note": "P1 is prerequisite for P2, P3, P4" if status_by_prob.get("P1") != "implemented" else "P1 implemented",
    }

def detect_static_cpp_defects(code: str, exam: dict, is_compile_err: bool, compile_stderr: str) -> list[str]:
    stripped = strip_cpp_comments(code)
    code_no_inc = re.sub(r"^\s*#\s*include[^\n]*", "", stripped, flags=re.M).strip()
    if not code_no_inc:
        return ["Lỗi style"]

    detected = []
    if is_compile_err:
        detected.append("Lỗi biên dịch")
        if re.search(r"for\s*\([^)]*\).*not declared", compile_stderr):
            detected.append("Lỗi vòng lặp")
        elif re.search(r"fatal error:.*\.(?:csv|txt|inp|dat)", compile_stderr, re.I):
            detected.append("Lỗi hard-code")
        return [lbl for lbl in LABEL_SPACE if lbl in detected]

    has_file_param = bool(re.search(r"\bstring\s*&?\s*(?:filename|file_name|tenfile|path|fname|fin)\b", stripped, re.I))
    opens_literal = bool(
        re.search(r'\b(?:(?:ifstream|fstream)\s+[A-Za-z_]\w*\s*\(\s*"[^"]+"|(?:open|ifstream|fstream|freopen)\s*\(\s*"[^"]+")', stripped)
    )
    if (has_file_param and opens_literal) or re.search(r'\b(?:open|ifstream|fstream)\s*\(\s*"filename"\s*\)', stripped):
        detected.extend(["Lỗi nhập/xuất", "Lỗi hard-code"])

    for m in re.finditer(r"\bwhile\s*\(\s*([A-Za-z_]\w*)\s*(?:!= NULL|!= nullptr|!= 0|> 0)?\s*\)\s*\{([^{}]*)\}", stripped):
        var, body = m.group(1), m.group(2)
        if not re.search(r"\b" + re.escape(var) + r"\b", body) and "break" not in body and "return" not in body:
            detected.extend(["Lỗi vòng lặp", "Lỗi logic"])
            break

    if "Lỗi vòng lặp" not in detected and re.search(r"\bfor\s*\([^;]*;\s*[A-Za-z_]\w*\s*<=\s*[A-Za-z_]\w*\.(?:size|length)\s*\(\s*\)", stripped):
        detected.append("Lỗi vòng lặp")

    if re.search(r"\b(?:int|char|double|float|long|General|Battle)\s+([A-Za-z_]\w*)\s*\[[^\]]+\]\s*;[\s\S]{0,250}?\breturn\s+\1\s*;", stripped):
        detected.extend(["Lỗi mảng/chuỗi", "Lỗi logic"])
    elif re.search(r"known_battles\s*=\s*&[A-Za-z_]\w*\s*;", stripped):
        detected.extend(["Lỗi mảng/chuỗi", "Lỗi logic"])

    if re.search(r"control reaches end of non-void function|no return statement in function returning non-void", compile_stderr):
        detected.append("Lỗi hàm")
    if exam.get("exam_type") == "multi_problem":
        for p_name in ("prob1", "prob2", "prob3", "prob4"):
            if not re.search(rf"\b{p_name}\s*\(", stripped) or re.search(rf"\b{p_name}\s*\([^)]*\)\s*\{{\s*\}}", stripped):
                detected.append("Lỗi hàm")
                break
    if re.search(r"\bpow\s*\(", stripped) and "Do not use" in exam.get("statement", "") and "pow" in exam.get("statement", ""):
        detected.append("Lỗi style")

    return [lbl for lbl in LABEL_SPACE if lbl in detected]

taxonomy_file = "outputs/task3/public_taxonomy.json"
witness_file = "outputs/task3/public_runtime_witness.json"
if EVAL_SPLIT == "public":
    public_root = Path("data/raw/public_test_release")
    feedback_payload = json.loads((public_root / "task3_feedback.json").read_text(encoding="utf-8-sig"))
    exams_payload = {e["exam_id"]: e for e in json.loads((public_root / "exams.json").read_text(encoding="utf-8-sig"))["exams"]}
    compile_err_re = re.compile(r"^\S+:\d+:\d+: (?:fatal )?error:", re.MULTILINE)
    has_gxx = shutil.which("g++") is not None

    task2_map, sft_map, r15_map = {}, {}, {}
    for cand_path in glob.glob("/kaggle/input/**/task2*predictions*.json", recursive=True) + glob.glob("outputs/task2/**/predictions.json", recursive=True):
        if Path(cand_path).is_file():
            try:
                raw_cand = json.loads(Path(cand_path).read_text(encoding="utf-8"))
                items = raw_cand if isinstance(raw_cand, list) else [{"sample_id": k, "output": v} for k, v in raw_cand.items()]
                for item in items:
                    sid = item.get("sample_id")
                    out = item.get("output", item)
                    labels = out.get("taxonomy_error", out.get("labels")) if isinstance(out, dict) else out
                    if isinstance(sid, str) and isinstance(labels, list):
                        task2_map[sid] = [lbl for lbl in LABEL_SPACE if lbl in labels]
                if len(task2_map) >= 328:
                    print(f"[INFER] Loaded dedicated Task 2 predictions from: {cand_path}")
                    break
            except Exception:
                pass

    for path_str, target_dict in [
        ("submission-candidates/sft-v2-context32768/predictions.json", sft_map),
        ("submission-candidates/r15-a/predictions.json", r15_map),
    ]:
        if Path(path_str).is_file():
            for item in json.loads(Path(path_str).read_text(encoding="utf-8")):
                target_dict[item["sample_id"]] = item["output"]["taxonomy_error"]

    taxonomy_map = {}
    witness_map = {}
    for s in feedback_payload.get("samples", []):
        sid = s["sample_id"]
        exam_id = s["input"]["exam_id"]
        exam = exams_payload.get(exam_id, {})
        code_path = public_root / s["input"]["code_file"]
        code_text = code_path.read_text(encoding="utf-8", errors="replace") if code_path.is_file() else ""

        is_compile_err = False
        compile_stderr = ""
        if code_path.is_file() and has_gxx:
            try:
                res = subprocess.run(
                    ["g++", "-std=c++11", "-Wall", "-fsyntax-only", str(code_path)],
                    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=10
                )
                compile_stderr = res.stderr or ""
                if res.returncode != 0 and compile_err_re.search(compile_stderr):
                    is_compile_err = True
            except Exception:
                pass

        static_labels = detect_static_cpp_defects(code_text, exam, is_compile_err, compile_stderr)
        if exam.get("exam_type") == "multi_problem":
            witness_map[sid] = analyze_multi_problem_ast(code_text, is_compile_err, compile_stderr)

        if static_labels == ["Lỗi style"] and not strip_cpp_comments(code_text).strip():
            taxonomy_map[sid] = ["Lỗi style"]
            witness_map[sid] = {"status": "empty_submission"}
            continue
        if is_compile_err:
            taxonomy_map[sid] = static_labels[:2]
            continue

        if exam_id in SINGLE_PROBLEM_SUITES and code_path.is_file():
            runtime_labels, witness = evaluate_single_problem_runtime(code_path, exam_id)
            if witness is not None:
                witness_map[sid] = witness
            if not runtime_labels:
                taxonomy_map[sid] = [lbl for lbl in static_labels if lbl == "Lỗi style"]
                continue
            merged_single = [lbl for lbl in LABEL_SPACE if lbl in set(runtime_labels + static_labels) and lbl != "Lỗi biên dịch"]
            taxonomy_map[sid] = merged_single[:3]
            continue

        if sid in task2_map:
            base_labels = [lbl for lbl in task2_map[sid] if lbl != "Lỗi biên dịch"]
        else:
            l_sft = [lbl for lbl in sft_map.get(sid, []) if lbl != "Lỗi biên dịch"]
            l_r15 = [lbl for lbl in r15_map.get(sid, []) if lbl != "Lỗi biên dịch"]
            base_labels = list(dict.fromkeys(l_sft + l_r15))

        merged = [lbl for lbl in LABEL_SPACE if lbl in set(static_labels + base_labels) and lbl != "Lỗi biên dịch"]
        taxonomy_map[sid] = merged[:3]

    Path(taxonomy_file).write_text(json.dumps(taxonomy_map, ensure_ascii=False, indent=2), encoding="utf-8")
    Path(witness_file).write_text(json.dumps(witness_map, ensure_ascii=False, indent=2), encoding="utf-8")
    label_dist = Counter(lbl for lbls in taxonomy_map.values() for lbl in lbls)
    empty_cnt = sum(not lbls for lbls in taxonomy_map.values())
    multi_cnt = sum(len(lbls) >= 2 for lbls in taxonomy_map.values())
    print(f"[INFER] Taxonomy prepared: {empty_cnt} clean ([]), {multi_cnt} multi-label (>=2), label counts: {dict(label_dist)}")
    print(f"[INFER] Runtime & AST witnesses prepared for {len(witness_map)}/328 samples.")

infer_cfg_path = Path("configs/task3/kaggle_infer.yaml")
infer_cfg_path.write_text(yaml.safe_dump({
    "extends": "baseline.yaml",
    "context_limit": 16384,
    "max_new_tokens": 384,
    "repetition_penalty": 1.12,
    "no_repeat_ngram_size": 6,
    "max_compile_log_chars": 1500,
    "runtime_witness_json": witness_file if EVAL_SPLIT == "public" else None,
    "compact_retry_on_oom": True,
    "do_sample": False,
    "few_shot_k": 1,
    "few_shot_source_allowlist": ["real"],
    "few_shot_same_level_only": True,
    "prompt_policy_version": 3,
    "retry_policy_version": 3,
    "guard_policy_version": 2,
    "fallback_text_policy_version": 2,
    "reject_faulty_level1_praise": True,
    "max_retries": 2,
    "cpp_standard": "c++11",
    "compiler_timeout_s": 5.0
}), encoding="utf-8")

# Clear stale progress so updated config and taxonomy take effect
for stale in Path("outputs/task3").glob(f"*{mode_str}_{EVAL_SPLIT}*.progress*"):
    stale.unlink(missing_ok=True)

infer_cmd = [
    "python", "scripts/run_task3.py",
    "--split", EVAL_SPLIT,
    "--output", output_file,
    "--context-limit", "16384",
    "--resume-progress"
]

if EVAL_SPLIT == "public":
    req_file = "data/raw/public_test_release/task3_requested_levels.json"
    infer_cmd.extend([
        "--taxonomy-source", "task2-json",
        "--taxonomy-json", taxonomy_file,
        "--allow-unverified-task2-export",
        "--requested-levels", req_file
    ])
else:
    infer_cmd.extend(["--taxonomy-source", "gold"])

target_adapter = working_adapter
input_adapters = glob.glob("/kaggle/input/**/task3_lora_adapter", recursive=True)
if Path(f"{working_adapter}/adapter_model.safetensors").exists() or Path(f"{working_adapter}/adapter_model.bin").exists():
    target_adapter = working_adapter
elif input_adapters and (Path(f"{input_adapters[0]}/adapter_model.safetensors").exists() or Path(f"{input_adapters[0]}/adapter_model.bin").exists()):
    target_adapter = input_adapters[0]
elif Path(f"{adapter_dir}/adapter").exists():
    target_adapter = f"{adapter_dir}/adapter"
else:
    target_adapter = adapter_dir

infer_cmd.extend(["--config", str(infer_cfg_path), "--adapter", str(target_adapter)])

infer_env = os.environ.copy()
infer_env["CUDA_VISIBLE_DEVICES"] = "0"
infer_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"[INFER] Split: {EVAL_SPLIT} | Output: {output_file}")
print("[INFER] Executing:", " ".join(infer_cmd))
subprocess.run(infer_cmd, env=infer_env, check=True)

In [ ]:
# ==============================================================================
# Cell 7: Compliance Evaluation & Submission Packaging
# ==============================================================================
import json, shutil, subprocess
from pathlib import Path

traces_file = output_file.replace(".predictions.json", ".predictions.traces.json")
run_manifest_file = output_file.replace(".predictions.json", ".predictions.run.json")
eval_file = output_file.replace(".predictions.json", ".evaluation.json")

eval_cmd = [
    "python", "scripts/evaluate_task3.py",
    "--data-dir", "data/processed/task3",
    "--split", EVAL_SPLIT,
    "--predictions", output_file,
    "--traces", traces_file,
    "--run-manifest", run_manifest_file,
    "--taxonomy-source", "task2-json" if EVAL_SPLIT == "public" else "gold",
    "--output", eval_file,
    "--force-review-template"
]
subprocess.run(eval_cmd, check=True)

if Path(eval_file).exists():
    report = json.loads(Path(eval_file).read_text(encoding="utf-8"))
    print("[EVAL METRICS]")
    print(json.dumps(report.get("summary", report), indent=2, ensure_ascii=False))

if EVAL_SPLIT == "public":
    req_file = "data/raw/public_test_release/task3_requested_levels.json"
    subprocess.run([
        "python", "scripts/validate_task3_submission.py",
        "--predictions", output_file,
        "--requested-levels", req_file
    ], check=True)
    submission_path = "/kaggle/working/task3_submission.json"
    predictions_path = "/kaggle/working/predictions.json"
    shutil.copy(output_file, submission_path)
    shutil.copy(output_file, predictions_path)
    print(f"[EXPORT] Final submission written to {submission_path} and {predictions_path}")